# Oxford Parkinson's Voice Biomarker Analysis & Classification Pipeline
### Multi-Disease Prediction System using Machine Learning (Minor Project Part-I)
**Dataset:** Oxford Parkinson's Telemonitoring Dataset (195 records, 22 acoustic voice features, status target)

---
### 1. Mathematical Formulation
* **Support Vector Machine (SVM) Optimization:**
  $$\min_{w, b, \xi} \frac{1}{2} ||w||^2 + C \sum_{i=1}^{n} \xi_i$$
  $$\text{subject to: } y_i (w^T \phi(x_i) + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$
* **Radial Basis Function (RBF) Kernel:**
  $$K(x, x') = \exp(-\gamma ||x - x'||^2)$$
* **Decision Tree Gini Impurity (CART):**
  $$I_G(t) = 1 - \sum_{k=1}^{C} p_k^2$$


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

### 2. Data Loading & Dropping Identifier Column

In [ ]:
data_path = os.path.join('Dataset minor', 'parkinsons.data')
df_raw = pd.read_csv(data_path)
df = df_raw.drop(columns=['name'])
print(f"Dataset Shape: {df.shape[0]} samples, {df.shape[1]-1} acoustic features + status")
display(df.head())

In [ ]:
print("Missing Values:", df.isnull().sum().sum())
print("\nStatus Counts:")
print(df['status'].value_counts())

### 3. Vocal Acoustic Biomarkers Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.boxplot(data=df, x='status', y='PPE', hue='status', ax=axes[0], palette=['#2980b9', '#c0392b'], legend=False)
axes[0].set_title("Pitch Period Entropy (PPE)", fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Healthy (0)', "Parkinson's (1)"])

sns.boxplot(data=df, x='status', y='spread1', hue='status', ax=axes[1], palette=['#2980b9', '#c0392b'], legend=False)
axes[1].set_title("Spread1 (Nonlinear Metric)", fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Healthy (0)', "Parkinson's (1)"])

sns.boxplot(data=df, x='status', y='HNR', hue='status', ax=axes[2], palette=['#2980b9', '#c0392b'], legend=False)
axes[2].set_title("Harmonics-to-Noise Ratio (HNR)", fontweight='bold')
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(['Healthy (0)', "Parkinson's (1)"])
plt.show()

### 4. Train/Test Split (80/20 Stratified) & Standardization

In [ ]:
X = df.drop('status', axis=1)
y = df['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training: {X_train.shape[0]} samples | Testing: {X_test.shape[0]} samples")

### 5. SVM Kernel Comparison & Zero False-Negative Screening

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
k_res = {}
for k in kernels:
    svm = SVC(kernel=k, C=1.0, probability=True, random_state=42)
    svm.fit(X_train_scaled, y_train)
    preds = svm.predict(X_test_scaled)
    probs = svm.predict_proba(X_test_scaled)[:, 1]
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    k_res[k.capitalize()] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Recall (Sensitivity)': recall_score(y_test, preds),
        'Specificity': tn / (tn + fp),
        'Precision': precision_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'False Negatives': fn
    }

display(pd.DataFrame(k_res).T)

### 6. Linear SVM vs Decision Tree Confusion Matrix (Slide 2 Alignment)

In [ ]:
svm_lin = SVC(kernel='linear', C=1.0, random_state=42).fit(X_train_scaled, y_train)
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(confusion_matrix(y_test, svm_lin.predict(X_test_scaled)), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Linear SVM (Zero False Negatives: 100% Recall!)", fontweight='bold')
axes[0].set_xticklabels(['Healthy', 'Parkinsons'])
axes[0].set_yticklabels(['Healthy', 'Parkinsons'])

sns.heatmap(confusion_matrix(y_test, dt.predict(X_test)), annot=True, fmt="d", cmap="Oranges", ax=axes[1])
axes[1].set_title("Decision Tree CART (3 Missed Cases)", fontweight='bold')
axes[1].set_xticklabels(['Healthy', 'Parkinsons'])
axes[1].set_yticklabels(['Healthy', 'Parkinsons'])
plt.show()

### 7. Decision Tree Architecture

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(dt, feature_names=list(X.columns), class_names=['Healthy', "Parkinson's"], filled=True, rounded=True, fontsize=9)
plt.title("Parkinson's Decision Tree Rules Flowchart", fontweight='bold')
plt.show()